In [1]:
fprs,tprs,precisions,recalls,normalization_roc,normalization_pr=[],[],[],[],[],[]


In [ ]:
#bash code

#normalA
find /root/sunxh/WaveCrossMamba/contrast/IVT_normalA_guppy_single/ -name "*.fast5" >normalA.txt
python /root/sunxh/nanom6A_2021_1_22/extract_raw_and_feature_fast.py --cpu=20 --fl=normalA.txt -o normalA --clip=10

python /root/sunxh/nanom6A_2022_12_22/predict_sites.py --cpu 20 -i normalA  -o normalA_final \
-r /root/sunxh/data/fna/ELIGOS_reference.fa -g /root/sunxh/data/fna/normal_anno.fa \
--model /root/sunxh/nanom6A_2022_12_22/bin/model

python /home/wuyou/Projects/nanom6A/nanom6A_2021_1_22/predict_sites_simple.py --cpu 20 -i normalA -o normalA_final \
-r /root/sunxh/data/fna/ELIGOS_reference.fa -g /root/sunxh/data/fna/normal_anno.fa \
--model /home/wuyou/Projects/nanom6A/nanom6A_2021_3_11/bin/model
samtools view -bS input.sam > output.bam

samtools faidx /root/sunxh/data/fna/m6a_anno.fa
picard CreateSequenceDictionary R=/root/sunxh/data/fna/m6a_anno.fa O=/root/sunxh/data/fna/m6a_anno.dict
#cat *.mod *.unmod >normalA.predict
#cat *.mod *.unmod >m6A.predict
cat /root/sunxh/nanom6A_2022_12_22/normalA_final/*.mod *.unmod >normalA.predict

samtools view -bS /root/sunxh/WaveCrossMamba/contrast/IVT_m6A.sam > /root/sunxh/WaveCrossMamba/contrast/m6A.bam
samtools sort /root/sunxh/WaveCrossMamba/contrast/m6A.bam -o /root/sunxh/WaveCrossMamba/contrast/m6a_output_sorted.bam
bedtools bamtobed -bed12 -i /root/sunxh/WaveCrossMamba/contrast/m6a_output_sorted.bam > /root/sunxh/WaveCrossMamba/contrast/m6a_output.bed
bedtools getfasta -fi /root/sunxh/nanom6A_2022_12_22/ELIGOS_reference.fa -bed /root/sunxh/WaveCrossMamba/contrast/m6a_output.bed -fo /root/sunxh/data/fna/m6a_anno.fa
samtools faidx ref.fa

#m6A
find /root/sunxh/WaveCrossMamba/contrast/IVT_m6A_guppy_single/ -name "*.fast5" >m6A.txt

python /root/sunxh/nanom6A_2022_12_22/extract_raw_and_feature_fast.py --cpu=20 --fl=m6A.txt -o m6A --clip=10

python /root/sunxh/nanom6A_2022_12_22/predict_sites.py --cpu 20 -i m6A  -o m6A_final \
-r /root/sunxh/nanom6A_2022_12_22/ELIGOS_reference.fa -g /root/sunxh/data/fna/m6a_anno.fa \
--model /root/sunxh/nanom6A_2022_12_22/bin/model

SyntaxError: invalid syntax (3538793872.py, line 4)

In [34]:
#calculate ROC and PR

import argparse
import traceback

import pandas as pd
import numpy as np
from scipy import interpolate

from sklearn import metrics
from sklearn import datasets
from sklearn.model_selection import train_test_split as ts
from sklearn.metrics import roc_curve,auc,roc_auc_score,precision_recall_curve

import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.autograd import Variable

from plotnine import *
import warnings
from sklearn.metrics import roc_curve,auc,roc_auc_score,precision_recall_curve
fprs,tprs,precisions,recalls,normalization_roc,normalization_pr=[],[],[],[],[],[]

batch_y,probabilities=[],[]
with open("/root/sunxh/nanom6A_2022_12_22/normalA_final/normalA.predict") as f:
    count=0
    for i,line in enumerate(f):
        if i%100==0:
            probability=float(line.split("\t")[1])
            batch_y.append(0)
            probabilities.append(probability)
            if count>500:
                break
with open("/root/sunxh/nanom6A_2022_12_22/normalA_final/m6A_final/m6A.predict") as f:
    count=0
    for i,line in enumerate(f):
        if i%100==0:
            probability=float(line.split("\t")[1])
            batch_y.append(1)
            probabilities.append(probability)
            if count>500:
                break
            
fpr,tpr,thersholds=roc_curve(batch_y,probabilities)
precision,recall,thersholds=precision_recall_curve(batch_y,probabilities)
roc_auc=auc(fpr,tpr)
pr_auc=auc(recall,precision)

fprs.extend(fpr)
tprs.extend(tpr)
precisions.extend(precision)
recalls.extend(recall)
normalization_roc.extend(["nanom6A AUC %.2f" %roc_auc]*len(fpr))
normalization_pr.extend(["nanom6A AUC %.2f" %pr_auc]*len(precision))
df_roc = pd.DataFrame({"FPR": fprs, "TPR": tprs, "AUC": normalization_roc})
df_roc.to_csv("nanom6a_roc.csv", sep=",", index=False)  
print("✅ 预测结果已保存到 `rrach_roc_results.csv` 和 `rrach_pr_results.csv`")


✅ 预测结果已保存到 `rrach_roc_results.csv` 和 `rrach_pr_results.csv`


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
df_roc = pd.read_csv("/root/sunxh/WaveCrossMamba/result_reproduce/nanom6a_roc.csv")
df_pr = pd.read_csv("/root/sunxh/WaveCrossMamba/result_reproduce/nanom6a_pr.csv")
fpr, tpr, auc_values_roc = df_roc["FPR"], df_roc["TPR"], df_roc["AUC"].iloc[0]
precision, recall, auc_values_pr = df_pr["Precision"], df_pr["Recall"], df_pr["AUC"].iloc[0]
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC Curve ({auc_values_roc})', linestyle='-', linewidth=2)
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')  
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, label=f'PR Curve ({auc_values_pr})', linestyle='-', linewidth=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc='upper right')
plt.grid(True)
plt.show()